# Trek Recommendation Engine: Cosine Similarity

This notebook builds a recommendation system to find similar treks using Cosine Similarity on the preprocessed trek features.

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data
We load the preprocessed datasets and scale the remaining features.

In [7]:
# Load preprocessed data
df_scaled = pd.read_csv('datasets/processed/trek_preprocessed_scaled.csv')
df_unscaled = pd.read_csv('datasets/processed/trek_preprocessed_unscaled.csv')

# Drop duplicates to ensure unique Trek names for the similarity matrix
df_scaled = df_scaled.drop_duplicates(subset=['Trek']).reset_index(drop=True)
df_unscaled = df_unscaled.drop_duplicates(subset=['Trek']).reset_index(drop=True)

# Scale Cost_USD as it was left unscaled in the pipeline
scaler = StandardScaler()
df_scaled['Cost_USD_scaled'] = scaler.fit_transform(df_scaled[['Cost_USD']])

# Select features for similarity matrix (drop string identifier and unscaled cost)
X = df_scaled.drop(columns=['Trek', 'Cost_USD'])
feature_cols = X.columns.tolist()

print(f"Dataset shape: {df_scaled.shape}")
print(f"Features used for recommendation: {feature_cols}")

Dataset shape: (68, 16)
Features used for recommendation: ['Duration_Days', 'Max_Altitude_m', 'Trip_Grade_Ordinal', 'Accom_Guesthouse', 'Accom_Lodge', 'Accom_LuxuryLodge', 'Accom_Teahouse', 'Accom_TeahouseLodge', 'Travel_AprMay_SeptNov', 'Travel_JanMay_SeptDec', 'Travel_MarMay_SeptDec', 'Travel_MarMay_SeptNov', 'Travel_MarNov', 'Cost_USD_scaled']


## 2. Compute Cosine Similarity Matrix
We calculate the pairwise cosine similarity between all treks.

In [8]:
# Compute similarity matrix
similarity_matrix = cosine_similarity(X.values)

# Create a DataFrame for easy lookup
similarity_df = pd.DataFrame(
    similarity_matrix, 
    index=df_scaled['Trek'], 
    columns=df_scaled['Trek']
)

print(f"Similarity matrix shape: {similarity_df.shape}")
similarity_df.iloc[:5, :5]

Similarity matrix shape: (68, 68)


Trek,Everest Base Camp Trek,Everest Base Camp Short Trek,Everest Base Camp Heli Shuttle Trek,Everest Base Camp Heli Trek,Everest Base Camp Trek for Seniors
Trek,,,,,
Everest Base Camp Trek,1.000000,0.959322,0.747406,0.361395,0.859012
Everest Base Camp Short Trek,0.959322,1.000000,0.753498,0.297955,0.682115
Everest Base Camp Heli Shuttle Trek,0.747406,0.753498,1.000000,0.844910,0.639760
Everest Base Camp Heli Trek,0.361395,0.297955,0.844910,1.000000,0.485454
Everest Base Camp Trek for Seniors,0.859012,0.682115,0.639760,0.485454,1.000000


## 3. Recommendation Function
We define a function that takes a trek name and returns the top N most similar treks based on the calculated similarity matrix.

In [9]:
def recommend_treks(trek_name, top_n=5):
    """
    Given a trek name, returns the top-N most similar treks based on features.
    """
    if trek_name not in similarity_df.index:
        print(f"Trek '{trek_name}' not found in the dataset.")
        return None
        
    # Get similarity scores for the given trek, exclude the trek itself
    similarities = similarity_df[trek_name].drop(trek_name).sort_values(ascending=False)
    top_similar = similarities.head(top_n)
    
    print(f"\n🏔️ Treks similar to '{trek_name}':")
    print("=" * 70)
    
    source_scaled = df_scaled[df_scaled['Trek'] == trek_name].iloc[0]
    
    # We will look at a few interpretable continuous features to provide explanations
    explain_features = ['Duration_Days', 'Max_Altitude_m', 'Cost_USD_scaled', 'Trip_Grade_Ordinal']
    explain_labels = ['Duration', 'Altitude', 'Cost', 'Difficulty']
    
    results = []
    for rank, (rec_trek, score) in enumerate(top_similar.items(), 1):
        rec_row_unscaled = df_unscaled[df_unscaled['Trek'] == rec_trek].iloc[0]
        rec_row_scaled = df_scaled[df_scaled['Trek'] == rec_trek].iloc[0]
        
        # Generate explanation by finding features with similar scaled values
        shared_traits = []
        for feat, label in zip(explain_features, explain_labels):
            if abs(source_scaled[feat] - rec_row_scaled[feat]) < 0.5:
                shared_traits.append(label)
                
        explanation = f"Similar in: {', '.join(shared_traits)}" if shared_traits else "Overall profile match"
        
        cost = rec_row_unscaled['Cost_USD']
        duration = rec_row_unscaled['Duration_Days']
        altitude = rec_row_unscaled['Max_Altitude_m']
        
        print(f"  {rank}. {rec_trek}")
        print(f"     Similarity: {score:.3f} | Cost: ${cost:.0f} | {duration:.0f} days | Max Alt: {altitude:.0f}m")
        print(f"     {explanation}")
        
        results.append({
            'Trek': rec_trek, 
            'Similarity': score, 
            'Cost_USD': cost,
            'Duration_Days': duration,
            'Max_Altitude_m': altitude
        })
        
    return pd.DataFrame(results)

## 4. Test Recommendations
Let's test the recommendation engine on a few popular treks.

In [10]:
demo_treks = ['Everest Base Camp Trek', 'Annapurna Circuit Trek', 'Mardi Himal Trek']

for trek in demo_treks:
    if trek in similarity_df.index:
        recommend_treks(trek)
    else:
        # Fallback to the first available trek if demo names are slightly different in dataset
        print(f"Trek {trek} not found. Here is a generic demo:")
        recommend_treks(df_scaled['Trek'].iloc[0])


🏔️ Treks similar to 'Everest Base Camp Trek':
  1. Gokyo Lake Renjo La Pass Trek
     Similarity: 0.995 | Cost: $1450 | 16 days | Max Alt: 5360m
     Similar in: Duration, Altitude, Cost, Difficulty
  2. Everest Base Camp Short Trek
     Similarity: 0.959 | Cost: $1295 | 14 days | Max Alt: 5545m
     Similar in: Duration, Altitude, Cost, Difficulty
  3. Everest Base Camp Trek for Seniors
     Similarity: 0.859 | Cost: $1800 | 20 days | Max Alt: 5545m
     Similar in: Altitude, Difficulty
  4. Tamang Heritage Trek
     Similarity: 0.779 | Cost: $920 | 14 days | Max Alt: 5050m
     Similar in: Duration, Altitude
  5. Langtang Gosaikunda Trek
     Similarity: 0.776 | Cost: $1180 | 16 days | Max Alt: 4600m
     Similar in: Duration, Cost

🏔️ Treks similar to 'Annapurna Circuit Trek':
  1. Langtang Valley Trek
     Similarity: 0.804 | Cost: $750 | 10 days | Max Alt: 5050m
     Similar in: Altitude
  2. Annapurna Base Camp Short Trek
     Similarity: 0.690 | Cost: $1090 | 11 days | Max Alt: